# Модуль 11.5. Дизайн инструментов: лесоруб и хорошие tools на smolagents

Этот ноутбук — практика к лекции «Дизайн инструментов: что значит „хороший tool"» (модуль 11.5 на сайте курса). Правило из лекции — «tool — это контракт, который читает модель»; здесь вы чините tools руками.

Сквозной герой — тот же, что в лекции: агент-лесоруб в мок-лесу. Лес — чистый Python-класс `Forest` (сетка 5×5, узлы дерева и камня, рюкзак, склад, кулдаун), движок — smolagents из Модуля 10. Мы соберём два интерфейса к одному и тому же миру: антигероя `do_action(action, target)` — один универсальный tool на всё — и героя из четырёх инструментов по одной цели: `move`, `gather`, `deposit`, `get_map`. А потом измерим разницу.

**Что вы получите на выходе:**

- мок-лес из лекции — чистый Python, без ключей и без сети;
- два набора tools на smolagents: «плохой» god-tool через `@tool` и «хороший» набор, собранный по арке лекции — имя → `enum` → обучающая ошибка → конверт → идемпотентность;
- детерминированный харнесс, который гоняет скриптового «агента» по обоим наборам и считает долю успешных вызовов, восстановление после ошибки и потерянные ходы;
- мини-линтер по чек-листу A→H, читающий спецификацию прямо из smolagents-объектов (`tool.name` / `tool.description` / `tool.inputs`).

**Карта ноутбука:**

- **Блок 1** — мок-лес + два набора tools на smolagents.
- **Блок 2 (ядро, keyless)** — харнесс bad-vs-good + линтер A→H.
- **Блок 3 (опционально, нужен `HF_TOKEN`)** — живая модель через `ToolCallingAgent` решает задачу лесоруба.
- **Блок 4 (опционально)** — те же принципы против живого API игры `https://kindomklaster.com`.
- **Задачи** — переписать спрятанный god-сценарий, сделать ошибку обучающей, добавить idempotency-key, перегнать линтер до 8/8.

Главное про запуск: ноутбук исполняется **целиком и без единого ключа** (`Run all`, без правок). Интернет нужен один раз — поставить smolagents. Блоки 3 и 4 делают мягкий пропуск (soft-skip), если ключа или сети нет, — keyless-прогон остаётся зелёным. Активность для сдачи — в финальной секции «Задачи».

## Подготовка окружения

Ставим две библиотеки: `smolagents` (движок из Модуля 10 — из него берём `@tool`, `Tool` и агентов) и `pydantic` (в ядре ноутбука не используется — smolagents собирает схемы сам, — но пригодится, если захотите переписать проверку аргументов своей моделью).

Честная пометка: **для установки нужен интернет**. В Colab он есть всегда; в Kaggle включите `Notebook options → Internet → On` (нужен phone-verified аккаунт). После установки весь базовый трек работает без сети. API-ключи не нужны совсем — ни здесь, ни в Блоках 1–2, ни в Задачах.

In [ ]:
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import smolagents  # noqa: F401
except Exception:
    _pip_install("smolagents")
    import smolagents  # noqa: F401

try:
    import pydantic  # noqa: F401
except Exception:
    _pip_install("pydantic")
    import pydantic  # noqa: F401

print("smolagents", smolagents.__version__, "| pydantic", pydantic.VERSION)

## Блок 1. Мок-лес и два набора tools

Сначала — мир, дословно из лекции. Чтобы пощупать дизайн tools, не нужен сервер: достаточно маленькой модели мира на чистом Python.

- сетка **5×5**, координаты `0..4`;
- узлы ресурсов: дерево на `(1, 2)` и `(3, 1)`, камень на `(2, 4)`;
- лесоруб и склад — на `(0, 0)`;
- рюкзак с лимитом **5**;
- **кулдаун** после каждого действия — повтор раньше времени отбивается ошибкой.

Одно отличие от лекции: там `COOLDOWN = 1.0`, здесь — `0.3`. Правила не меняются ни на йоту, просто `Run all` со всеми честными паузами укладывается в минуту, а не в пять. Этот один мир мы дальше подставим под оба набора tools — разница будет только в интерфейсе, не в правилах.

In [ ]:
import time


class Forest:
    """Мок-лес: сетка 5x5, лесоруб, рюкзак, склад и кулдаун."""

    COOLDOWN = 0.3  # в лекции 1.0; здесь меньше, чтобы Run all занимал ~минуту, — правила те же

    def __init__(self):
        self.size = 5
        self.nodes = {(1, 2): "wood", (3, 1): "wood", (2, 4): "stone"}
        self.pos = (0, 0)        # где стоит лесоруб
        self.home = (0, 0)       # клетка склада
        self.backpack = {}       # например, {"wood": 3}
        self.cap = 5             # вместимость рюкзака
        self.stock = {}          # что уже сдано на склад
        self.busy_until = 0.0    # когда закончится кулдаун

    def cooldown_left(self):
        return max(0.0, self.busy_until - time.monotonic())

    def start_cooldown(self):
        self.busy_until = time.monotonic() + self.COOLDOWN

    def backpack_load(self):
        return sum(self.backpack.values())


forest = Forest()


def reset_forest():
    """Свежий мир перед каждым сценарием. Tools ниже смотрят на глобальную forest."""
    global forest
    forest = Forest()


print("Лес собран:", forest.size, "x", forest.size,
      "| узлы:", forest.nodes, "| лесоруб и склад на", forest.home)

### Антигерой: god-tool `do_action`

Первый интерфейс к этому миру — тот самый, из-за которого агенты «тупят». Один инструмент на всё: `do_action(action, target)`, обе строки свободные, любой отказ — `{"error": "invalid"}`.

Пишем его через `@tool` — так же, как вы писали инструменты в Модуле 10. Обратите внимание: docstring честный, блок `Args:` на месте, smolagents соберёт из этого схему без единой жалобы. Формально это валидный tool — и в этом весь ужас: валидная схема не спасает от плохого контракта.

Внутри — спрятанный `switch` по строке. Добыча называется `harvest_node`, ходьба — `relocate` с однобуквенными направлениями, сдача на склад — `store_items`, и есть ещё один тайный глагол, до которого мы доберёмся в Задаче 1. Модель этих имён не видит — ей остаётся угадывать.

In [ ]:
from smolagents import tool, Tool


@tool
def do_action(action: str, target: str) -> dict:
    """Выполнить действие в лесу.

    Args:
        action: какое действие выполнить.
        target: цель действия.
    """
    if forest.cooldown_left() > 0:
        return {"error": "invalid"}                  # кулдаун есть, но tool о нём молчит

    if action == "harvest_node" and target == "self_tile":   # внутренние имена!
        node = forest.nodes.get(forest.pos)
        if node is None or forest.backpack_load() >= forest.cap:
            return {"error": "invalid"}              # пусто? рюкзак полон? — не узнать
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"ok": True}

    if action == "relocate" and target in ("n", "s", "e", "w"):
        dx, dy = {"n": (0, -1), "s": (0, 1), "e": (1, 0), "w": (-1, 0)}[target]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        forest.start_cooldown()
        return {"ok": True}

    if action == "store_items" and target == "depot":
        if forest.pos != forest.home or not forest.backpack:
            return {"error": "invalid"}
        for res, n in forest.backpack.items():
            forest.stock[res] = forest.stock.get(res, 0) + n
        forest.backpack = {}
        forest.start_cooldown()
        return {"ok": True}

    if action == "assemble" and target == "axe_kit":
        # тайный сценарий — крафт топора из 3 wood на складе (Задача 1)
        if forest.pos != forest.home or forest.stock.get("wood", 0) < 3:
            return {"error": "invalid"}
        forest.stock["wood"] -= 3
        forest.stock["axe"] = forest.stock.get("axe", 0) + 1
        forest.start_cooldown()
        return {"ok": True}

    return {"error": "invalid"}                      # неизвестный глагол — и снова немой отказ


print("Плохой набор: один tool на всё —", do_action.name)

Убедимся, что этот tool ставит модель в положение угадывания. Прогоним вызовы, какие сделала бы модель, впервые увидевшая `do_action`: она хочет срубить дерево и пробует глаголы.

Заодно распечатаем контракт глазами smolagents — те четыре поля, которые движок положит в system prompt. Всё выглядит прилично: имя есть, описание есть, аргументы описаны. Проблема не в форме, а в содержании: из контракта не следует ни один допустимый вызов.

In [ ]:
reset_forest()

def wait_ready():
    """Честно подождать конец кулдауна (наш харнесс, не часть контракта)."""
    time.sleep(forest.cooldown_left())

# модель хочет срубить дерево и угадывает строки
print('do_action("cut", "big tree")  ->', do_action(action="cut", target="big tree"))
print('do_action("chop", "tree")     ->', do_action(action="chop", target="tree"))
print('do_action("harvest", "wood")  ->', do_action(action="harvest", target="wood"))
# даже угадав внутренний глагол, модель промахивается: рубить можно только под ногами
print('do_action("harvest_node", "self_tile") ->',
      do_action(action="harvest_node", target="self_tile"))
print()
print("Четыре разных промаха — один немой ответ. Ветвиться не по чему, остаётся перебор.")
print()
print("Контракт do_action глазами smolagents:")
print("  name:       ", do_action.name)
print("  description:", do_action.description)
print("  inputs:     ", do_action.inputs)
print("  output_type:", do_action.output_type)

### Шаг 1. Имена по одной цели: наивная версия через `@tool`

Чиним первое звено контракта — имя. Вместо одного `do_action` дадим лесорубу инструменты по одному на цель: `move`, `gather`, `get_map` (а `deposit` допишем чуть позже, когда дойдём до конверта ответа). Пока — в самом простом виде, через `@tool`, дословно как в лекции.

Правило из Модуля 10 работает и здесь: в 80% случаев хватает декоратора. smolagents прочитает аннотации типов и docstring и соберёт контракт сам — мы его сразу распечатаем.

И сразу увидим цену наивности: `gather` без проверок на пустой клетке добудет `None` и положит его в рюкзак. Ошибки и проверки — впереди, по арке лекции.

In [ ]:
@tool
def move(direction: str) -> dict:
    """Шаг на одну клетку в сторону direction.

    Args:
        direction: куда шагнуть — north, south, east или west.
    """
    dx, dy = {"north": (0, -1), "south": (0, 1),
              "east": (1, 0), "west": (-1, 0)}[direction]
    x, y = forest.pos
    forest.pos = (min(max(x + dx, 0), forest.size - 1),
                  min(max(y + dy, 0), forest.size - 1))
    return {"result": {"pos": list(forest.pos)}}


@tool
def gather() -> dict:
    """Добыть один ресурс (дерево или камень) с клетки, на которой стоит лесоруб."""
    node = forest.nodes.get(forest.pos)          # "wood" | "stone" | None
    forest.backpack[node] = forest.backpack.get(node, 0) + 1
    return {"result": {"gathered": node}}        # пока наивно — проверки добавим дальше


@tool
def get_map() -> dict:
    """Карта леса: позиция лесоруба, узлы ресурсов и клетка склада. Мир не меняет."""
    return {"pos": list(forest.pos), "home": list(forest.home),
            "nodes": [{"pos": list(p), "resource": r} for p, r in forest.nodes.items()]}


# Контракт глазами модели — всё, что она увидит про move:
print("name:       ", move.name)
print("description:", move.description)
print("inputs:     ", move.inputs)
print("output_type:", move.output_type)

# И цена наивности: gather без проверок кладёт None в рюкзак
reset_forest()
print()
print("наивный gather на пустой клетке ->", gather())
print("рюкзак:", forest.backpack, " <- вот зачем дальше нужны проверки и ошибки")
reset_forest()

### Шаг 2. `enum` в схеме: subclass `Tool`

`direction: str` — свободная строка. Docstring просит четыре слова, но docstring — просьба, а не рельсы: модель вольна прислать `"up"`, `"North"` или `"вверх"`. Нужен `enum` — закрытый список значений прямо в схеме аргумента, а это первый случай из правила Модуля 10, когда декоратора мало и нужен subclass `Tool`.

Что требует subclass (мы проверили интроспекцией на реальной версии smolagents):

- обязательные атрибуты: `name`, `description`, `inputs`, `output_type` и метод `forward`;
- каждый вход в `inputs` обязан иметь ключи `type` и `description` — иначе `AssertionError` при создании;
- дополнительные ключи вроде `enum` и `nullable` валидатор пропускает и кладёт в схему как есть;
- имена параметров `forward` должны совпадать с ключами `inputs`, а у параметра с дефолтом в `inputs` должен стоять `"nullable": True`.

Куда попадает `enum`, зависит от типа агента: `ToolCallingAgent` отдаёт `inputs` в JSON-схему как есть (сейчас это увидим), а `CodeAgent` рендерит инструмент Python-стабом — поэтому дубль списка в description («north, south, east или west») обязателен, это не подстраховка.

In [ ]:
import json


class MoveTool(Tool):
    name = "move"
    description = (
        "Шаг на одну клетку в сторону direction. "
        "Зовите, когда до нужного узла или склада не хватает шага."
    )
    inputs = {
        "direction": {
            "type": "string",
            "enum": ["north", "south", "east", "west"],
            "description": "Куда шагнуть: north, south, east или west.",
        }
    }
    output_type = "object"

    def forward(self, direction: str) -> dict:
        dx, dy = {"north": (0, -1), "south": (0, 1),
                  "east": (1, 0), "west": (-1, 0)}[direction]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        # пока без кулдауна и конверта — допишем по арке лекции ниже
        return {"result": {"pos": list(forest.pos)}}


move = MoveTool()   # экземпляр Tool зовётся так же, как раньше звалась функция
print("enum в контракте:", move.inputs["direction"]["enum"])
print()

# Что уедет в JSON-схему инструмента для ToolCallingAgent:
try:
    from smolagents.models import get_tool_json_schema
    print(json.dumps(get_tool_json_schema(move), ensure_ascii=False, indent=2))
except ImportError:
    print("(в этой версии smolagents хелпер называется иначе — не страшно: enum виден в move.inputs выше)")

# Жёсткой проверки значений на входе smolagents не делает — значение вне enum дойдёт до логики:
print()
try:
    move(direction="up")
except KeyError as e:
    print("move(direction='up') дошёл до forward и упал:", repr(e))
    print("enum работает ДО вызова (модель видит список и не промахивается),")
    print("а редкий оставшийся промах должна ловить логика tool и её ошибка — следующий шаг.")

### Шаг 3. Обучающие ошибки `{code, message}`

Ядро лекции. Хорошая ошибка называет проблему и подсказывает выход: машиночитаемый `code` — для ветвления, человекочитаемый `message` — что не так и что делать дальше. Весь язык ошибок лесоруба — четыре кода:

| Ситуация | code | message |
|---|---|---|
| `gather` на клетке без узла | `no_resource_here` | «На этой клетке нет нужного узла — найдите его через get_map и подойдите move.» |
| `gather` с полным рюкзаком | `inventory_full` | «Рюкзак полон (5/5) — вернитесь на склад (0, 0) и позовите deposit.» |
| `deposit` не на клетке склада | `not_at_storehouse` | «Склад на клетке (0, 0), а вы — на (3, 1). Дойдите до склада и повторите deposit.» |
| любое действие во время кулдауна | `on_cooldown` | «Лесоруб занят ещё 0.2 с — подождите и повторите.» |

Прочтите их как инструкции, а не жалобы: каждая говорит, что сделать следующим ходом. Соберём `gather` целиком — subclass с `enum`-аргументом `resource` (необязательная страховка-ожидание: указали `wood`, а стоите на камне — честный отказ вместо «добыл не то») и тремя проверками из лекции.

In [ ]:
class GatherTool(Tool):
    name = "gather"
    description = (
        "Добыть один ресурс с клетки, на которой стоит лесоруб. "
        "Без resource берёт то, что есть на клетке; с resource — только ожидаемое."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood", "stone"],
            "nullable": True,
            "description": "Ожидаемый ресурс: wood или stone. По умолчанию — любой.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        node = forest.nodes.get(forest.pos)
        if node is None or (resource is not None and node != resource):
            return {"error": {"code": "no_resource_here",
                              "message": "На этой клетке нет нужного узла — "
                                         "найдите его через get_map и подойдите move."}}
        if forest.backpack_load() >= forest.cap:
            return {"error": {"code": "inventory_full",
                              "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — "
                                         "вернитесь на склад (0, 0) и позовите deposit."}}
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"result": {"gathered": node}}   # конверт ответа доделаем в следующей секции


gather = GatherTool()
reset_forest()
print("gather на пустой клетке (0, 0):")
print(json.dumps(gather(), ensure_ascii=False, indent=2))
print()
print("Сравните с немым invalid: здесь есть и code для ветвления, и следующий шаг текстом.")

### Шаги 4–5. Конверт `{result, cooldown, state}` и кулдаун везде

Осталось два звена. Успешный ответ — не проза («Вы успешно добыли древесину!»), а один и тот же конверт из трёх полей:

- `result` — что именно произошло, машиночитаемо;
- `cooldown` — через сколько секунд можно действовать снова (поле прямо под следующее решение агента);
- `state` — свежее состояние лесоруба, чтобы модели не нужен был лишний вызов `get_map`.

И идемпотентность: после любого мутирующего действия лесоруб «занят». Повтор на ретрае не дублирует эффект, а отбивается понятной ошибкой `on_cooldown`. В `gather` кулдаун уже стоит; допишем его и в `move` — это ровно микропроверка из лекции («допишите в move тот же кулдаун»). Чтение `get_map` идемпотентно по природе — его не трогаем.

Финализируем весь хороший набор: `gather` и `move` получают полный конверт, `deposit` пишем целиком по лекции.

In [ ]:
class GatherTool(Tool):
    name = "gather"
    description = (
        "Добыть один ресурс с клетки, на которой стоит лесоруб. "
        "Без resource берёт то, что есть на клетке; с resource — только ожидаемое."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood", "stone"],
            "nullable": True,
            "description": "Ожидаемый ресурс: wood или stone. По умолчанию — любой.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        node = forest.nodes.get(forest.pos)
        if node is None or (resource is not None and node != resource):
            return {"error": {"code": "no_resource_here",
                              "message": "На этой клетке нет нужного узла — "
                                         "найдите его через get_map и подойдите move."}}
        if forest.backpack_load() >= forest.cap:
            return {"error": {"code": "inventory_full",
                              "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — "
                                         "вернитесь на склад (0, 0) и позовите deposit."}}
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"result": {"gathered": node, "amount": 1},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


class MoveTool(Tool):
    name = "move"
    description = (
        "Шаг на одну клетку в сторону direction. "
        "Зовите, когда до нужного узла или склада не хватает шага."
    )
    inputs = {
        "direction": {
            "type": "string",
            "enum": ["north", "south", "east", "west"],
            "description": "Куда шагнуть: north, south, east или west.",
        }
    }
    output_type = "object"

    def forward(self, direction: str) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        dx, dy = {"north": (0, -1), "south": (0, 1),
                  "east": (1, 0), "west": (-1, 0)}[direction]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        forest.start_cooldown()
        return {"result": {"pos": list(forest.pos)},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


@tool
def deposit() -> dict:
    """Сдать содержимое рюкзака на склад. Работает только на клетке склада (0, 0)."""
    wait = forest.cooldown_left()
    if wait > 0:
        return {"error": {"code": "on_cooldown",
                          "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке (0, 0), а вы — на {forest.pos}. "
                                     "Дойдите до склада и повторите deposit."}}
    banked, forest.backpack = forest.backpack, {}
    for res, n in banked.items():
        forest.stock[res] = forest.stock.get(res, 0) + n
    forest.start_cooldown()
    return {"result": {"banked": banked},
            "cooldown": forest.COOLDOWN,
            "state": {"pos": list(forest.pos), "backpack": {}, "stock": forest.stock}}


move, gather = MoveTool(), GatherTool()
GOOD_TOOLS = {"move": move, "gather": gather, "deposit": deposit, "get_map": get_map}
print("Хороший набор:", list(GOOD_TOOLS))

# конверт в деле: дойти до дерева (1, 2) и добыть
reset_forest()
for d in ("east", "south", "south"):
    wait_ready()
    move(direction=d)
wait_ready()
print(json.dumps(gather(resource="wood"), ensure_ascii=False, indent=2))

Сравним оба набора на одном и том же промахе — «добыть на пустой клетке». Одинаковая ситуация, одинаковый мир — разные ответы. Это самый показательный кадр лекции.

In [ ]:
reset_forest()
print("ПЛОХОЙ  gather на пустой клетке:", do_action(action="harvest_node", target="self_tile"))
reset_forest()
resp = gather()
print("ХОРОШИЙ gather на пустой клетке:", resp["error"]["code"], "—", resp["error"]["message"])
print()
print("Из плохого ответа модель не знает, что не так: глагол? цель? позиция?")
print("Из хорошего — знает и чинится за один ход: get_map -> move к дереву -> gather.")

## Блок 2 (ядро, keyless). Харнесс bad-vs-good и линтер A→H

Это центр ноутбука, и он работает без ключей. Две вещи:

1. **Детерминированный харнесс.** Скриптовый «агент» с фиксированной стратегией гоняет один сценарий — «найди дерево → дойди → руби до `inventory_full` → вернись → `deposit`» — через оба набора. Никакой LLM: стратегия зашита, чтобы цифры были воспроизводимы. Метрики: доля успешных вызовов, восстановление после ошибки (агент прочитал `code` и сделал осмысленный следующий ход) и потерянные ходы (вызовы, из которых агент не извлёк ничего).

2. **Линтер A→H.** Правила чек-листа как код. Спецификацию он читает **прямо из smolagents-объектов** — `tool.name`, `tool.description`, `tool.inputs` — то есть ровно те поля, которые видит модель.

На «хорошем» наборе агент допускает ошибки нарочно (первый `gather` до похода к дереву, добыча до упора в лимит) — но каждая ошибка конвертируется в следующий ход, потому что сама его называет. Благодаря `COOLDOWN = 0.3` весь прогон с честными паузами занимает секунды.

In [ ]:
def path_steps(frm, to):
    """Список направлений из frm в to: сначала выравниваем x, потом y."""
    steps = []
    dx, dy = to[0] - frm[0], to[1] - frm[1]
    steps += ["east"] * max(dx, 0) + ["west"] * max(-dx, 0)
    steps += ["south"] * max(dy, 0) + ["north"] * max(-dy, 0)
    return steps


def run_good_scenario(verbose=False):
    """Скриптовый агент на ХОРОШЕМ наборе. Ошибки читает по code и чинится за один ход."""
    reset_forest()
    stats = {"calls": 0, "ok": 0, "recovered": 0}

    def call(tool_obj, **kwargs):
        wait_ready()
        resp = tool_obj(**kwargs)
        stats["calls"] += 1
        if "error" in resp:
            if verbose:
                print(f"  {tool_obj.name}{kwargs or '()'} -> ERROR {resp['error']['code']}")
        else:
            stats["ok"] += 1
            if verbose:
                print(f"  {tool_obj.name}{kwargs or '()'} -> OK {resp.get('result', resp)}")
        return resp

    # 1) наивная попытка добыть прямо на старте — получаем обучающую ошибку
    resp = call(gather)
    assert resp["error"]["code"] == "no_resource_here"
    stats["recovered"] += 1        # ошибка сама сказала следующий шаг: get_map -> move

    # 2) находим ближайшее дерево по карте и идём к нему
    m = call(get_map)
    target = min((tuple(n["pos"]) for n in m["nodes"] if n["resource"] == "wood"),
                 key=lambda p: abs(p[0] - m["pos"][0]) + abs(p[1] - m["pos"][1]))
    for d in path_steps(tuple(m["pos"]), target):
        call(move, direction=d)

    # 3) рубим, пока не упрёмся в лимит рюкзака
    while True:
        resp = call(gather, resource="wood")
        if "error" in resp:
            assert resp["error"]["code"] == "inventory_full"
            stats["recovered"] += 1    # ошибка говорит: на склад и deposit
            break

    # 4) возвращаемся и сдаём
    for d in path_steps(target, (0, 0)):
        call(move, direction=d)
    resp = call(deposit)
    stats["stock"] = dict(resp["result"]["banked"])
    stats["lost"] = (stats["calls"] - stats["ok"]) - stats["recovered"]
    return stats


print("Хороший набор, сценарий 'найди дерево -> добудь до отказа -> сдай':")
res_good = run_good_scenario(verbose=True)
print()
print("Итог good:", res_good)

Теперь тот же сценарий на «плохом» наборе. «Агент» вынужден угадывать строки: глаголы `do_action`, формат `target`, однобуквенные направления. Главное — когда он промахивается, `{"error": "invalid"}` не несёт `code`, по которому можно ветвиться. Каждая ошибка — потерянный ход: агенту нечего прочитать, остаётся слепой перебор.

Последовательность ниже — те же намерения, что у хорошего агента, но так, как их прислала бы модель, не знающая внутренних имён.

In [ ]:
def run_bad_scenario(verbose=False):
    """Скриптовый агент на ПЛОХОМ наборе: угадывание строк, немые ошибки, слепые повторы."""
    reset_forest()
    stats = {"calls": 0, "ok": 0, "recovered": 0, "lost": 0}

    plan = [
        ("cut", "big tree"),            # угадывание глагола
        ("chop", "tree"),               # синоним — тоже мимо
        ("harvest", "wood"),            # ещё синоним
        ("harvest_node", "self_tile"),  # глагол угадан! но клетка пустая — тот же invalid
        ("relocate", "east"),           # формат цели не тот (нужна одна буква)
        ("relocate", "e"),              # угадан: (0,0) -> (1,0)
        ("relocate", "s"),              # (1,1)
        ("relocate", "s"),              # (1,2) — под ногами дерево
        ("harvest_node", "self_tile"),
        ("harvest_node", "self_tile"),
        ("harvest_node", "self_tile"),
        ("harvest_node", "self_tile"),
        ("harvest_node", "self_tile"),  # рюкзак полон (5/5)
        ("harvest_node", "self_tile"),  # invalid — но почему? агент не знает
        ("harvest_node", "self_tile"),  # слепой повтор — снова invalid
        ("relocate", "w"),
        ("relocate", "n"),
        ("relocate", "n"),              # дома, (0,0)
        ("deposit", "depot"),           # угадывание глагола сдачи
        ("store_items", "depot"),       # найден
    ]
    for action, target in plan:
        wait_ready()
        resp = do_action(action=action, target=target)
        stats["calls"] += 1
        if "error" in resp:
            stats["lost"] += 1          # немая ошибка: извлечь из неё нечего
            if verbose:
                print(f"  do_action({action!r}, {target!r}) -> {resp}")
        else:
            stats["ok"] += 1
            if verbose:
                print(f"  do_action({action!r}, {target!r}) -> OK")
    stats["stock"] = dict(forest.stock)
    return stats


print("Плохой набор, те же намерения:")
res_bad = run_bad_scenario(verbose=True)
print()
print("Итог bad:", res_bad)

Сведём цифры. Смотрите не на абсолютные числа (они зависят от сценария), а на **разрыв**: на хорошем наборе доля успеха выше, все ошибки восстановимые, потерянных ходов ноль. На плохом каждая ошибка — тупик.

Заметьте: оба агента в итоге сдали 5 дерева — мир один и тот же, и задача в принципе решаема даже через god-tool. Разница в цене: плохой набор потратил на неё треть вызовов впустую, и это скриптовый агент, который «угадывает» по нашему сценарию с первой-второй попытки. Живая модель перебирает дольше — каждый потерянный ход у неё стоит реальных токенов и шага агентного цикла.

In [ ]:
def pct(a, b):
    return f"{(100.0 * a / b):.0f}%" if b else "n/a"

rows = [
    ("всего вызовов",        res_bad["calls"],                     res_good["calls"]),
    ("успешных вызовов",     res_bad["ok"],                        res_good["ok"]),
    ("доля успеха",          pct(res_bad["ok"], res_bad["calls"]), pct(res_good["ok"], res_good["calls"])),
    ("ошибок восстановлено", res_bad["recovered"],                 res_good["recovered"]),
    ("потерянных ходов",     res_bad["lost"],                      res_good["lost"]),
    ("сдано на склад",       res_bad["stock"],                     res_good["stock"]),
]
print(f"{'метрика':<24}{'плохой':>16}{'хороший':>16}")
print("-" * 56)
for name, b, g in rows:
    print(f"{name:<24}{str(b):>16}{str(g):>16}")
print()
assert res_good["stock"].get("wood", 0) == 5, "хороший агент должен сдать полный рюкзак дерева"
assert res_bad["stock"].get("wood", 0) == 5, "мир один: плохой тоже доносит — но дороже"
assert res_good["lost"] == 0, "на хорошем наборе каждая ошибка конвертируется в следующий ход"
print("Разрыв — целиком заслуга интерфейса: мир и правила в обоих прогонах одинаковые.")

### Линтер A→H: правила как код

Чек-лист из лекции можно выразить кодом. Линтер ниже читает контракт **прямо из smolagents-объекта** — `tool.name`, `tool.description`, `tool.inputs` — ровно те поля, что видит модель. А поведение, которое из контракта кодом не достать (форма ошибок, конверт, идемпотентность, лимиты), декларирует «паспорт» `meta` — держите его честным по отношению к тому, что tool реально делает.

Восемь проверок:

- **A** имя — глагол по одной цели, `snake_case`, не god-tool (и без аргумента `action`);
- **B** description — есть и не «худая» однострочная заглушка;
- **C** аргументы — там, где у значения закрытый список, в схеме стоит `enum`;
- **D** возврат — конверт со структурой и полями под следующий шаг;
- **E** ошибки — `{code, message}`: что не так и что делать;
- **F** идемпотентность — мутация защищена (кулдаун или idempotency-key);
- **G** безопасность — узкий blast radius (нет `shell`/`eval`/произвольного http);
- **H** производительность — выдача ограничена (`limit`), дорогой tool помечен.

In [ ]:
GOD_NAMES = {"do_action", "do", "execute", "run", "action", "perform", "manage"}
DANGEROUS = {"shell", "run_code", "eval", "exec", "http_get", "http_get_any"}


def lint_tool(tool_obj, meta):
    """Оценить tool по чек-листу A->H: контракт — из smolagents-объекта, поведение — из meta."""
    name = tool_obj.name
    desc = (tool_obj.description or "").strip()
    inputs = tool_obj.inputs or {}

    checks = {
        "A_name": (name == name.lower() and name not in GOD_NAMES
                   and "action" not in inputs),
        "B_description": len(desc) >= 30,
        "C_args_enum": all("enum" in inputs.get(a, {})
                           for a in meta.get("closed_set_args", [])),
        "D_return_structured": (bool(meta.get("returns_envelope"))
                                and bool(meta.get("next_step_fields"))),
        "E_errors_teach": (bool(meta.get("error_has_code"))
                           and bool(meta.get("error_has_message"))),
        "F_idempotent": (not meta.get("is_mutation")
                         or meta.get("idempotency") in {"cooldown", "idempotency_key"}),
        "G_blast_radius": name not in DANGEROUS and not meta.get("arbitrary_exec"),
        "H_perf_limits": not meta.get("returns_list") or bool(meta.get("has_limit")),
    }
    score = sum(checks.values())
    return {"score": score, "max": len(checks), "checks": checks,
            "misses": [k for k, v in checks.items() if not v]}


def report(title, tool_obj, meta):
    r = lint_tool(tool_obj, meta)
    print(f"{title}: score {r['score']}/{r['max']}")
    if r["misses"]:
        print("  проседает:", ", ".join(r["misses"]))
    return r


print("Линтер A->H готов: 8 проверок, контракт читается из tool.name / tool.description / tool.inputs.")

Прогоним линтер на обоих героях Блока 1 — настоящих smolagents-объектах, а не пересказах. `do_action` соберёт мало (god-имя, свободные строки без `enum`, немые ошибки, ничего под следующий шаг), финальный `gather` — максимум.

Конкретные буквы, на которых проседает `do_action`, — это и есть «до» для домашних правок.

In [ ]:
meta_do_action = {
    "closed_set_args": ["action"],   # глаголы — закрытый список, но enum в контракте нет
    "returns_envelope": False,       # {"ok": True} / немой invalid
    "next_step_fields": [],
    "error_has_code": False,
    "error_has_message": False,
    "is_mutation": True,
    "idempotency": None,             # кулдаун внутри есть, но tool о нём молчит и его не объясняет
    "returns_list": False,
    "has_limit": False,
}

meta_gather = {
    "closed_set_args": ["resource"],
    "returns_envelope": True,        # конверт {result, cooldown, state}
    "next_step_fields": ["cooldown", "state"],
    "error_has_code": True,
    "error_has_message": True,
    "is_mutation": True,
    "idempotency": "cooldown",
    "returns_list": False,
    "has_limit": False,
}

print("=== ДО (god-tool) ===")
r_before = report("do_action", do_action, meta_do_action)
print()
print("=== ПОСЛЕ (узкий инструмент) ===")
r_after = report("gather", gather, meta_gather)
print()
print(f"Рост score: {r_before['score']} -> {r_after['score']} из {r_after['max']}")
assert r_after["score"] == r_after["max"], "финальный gather должен набирать максимум"

## Блок 3 (опционально, нужен `HF_TOKEN`). Живая модель решает задачу лесоруба

До сих пор «агентом» был скрипт — это давало воспроизводимые цифры. Теперь интереснее: отдать хороший набор **живой модели** и посмотреть, как контракт ведёт её к цели.

Нужен `HF_TOKEN` — тот самый токен Hugging Face, который вы завели в Модуле 1.5 и использовали в Модуле 10 (бесплатной квоты Inference Providers хватает). Блок **необязательный**: нет токена — ячейки печатают причину пропуска и не делают ни одной сетевой попытки, keyless-прогон остаётся зелёным. Токен кладите в переменную окружения или в Secrets платформы (Colab: `Secrets` слева; Kaggle: `Add-ons → Secrets`), никогда в код.

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

# В Colab/Kaggle токен часто лежит в Secrets — попробуем мягко достать.
if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    print("HF_TOKEN найден -> Блок 3 готов к запуску.")
else:
    print("HF_TOKEN не найден -> Блок 3 пропущен (keyless-прогон это норма).")

Запускаем `ToolCallingAgent` — второго агента smolagents, с JSON-вызовами вместо кода: он отдаёт наши `inputs` в схему как есть, так что модель видит `enum` заранее (вы распечатали эту схему в Блоке 1). Инструменты — те же четыре объекта, которые только что гонял скриптовый харнесс: контракту всё равно, кто его читает.

Две приземлённые детали. Во-первых, `max_steps` ограничен — задача решается за несколько шагов, а лимит бережёт квоту. Во-вторых, кулдаун 0.3 с здесь почти не мешает: пауза между шагами агента (запрос к модели) обычно длиннее, а если модель всё же поспешит — получит обучающий `on_cooldown` и повторит. Ровно так это работает и в реальных API с `Retry-After`.

Если хотите режим из Модуля 10 — замените `ToolCallingAgent` на `CodeAgent`: контракт тот же, только модель будет писать вызовы кодом.

In [ ]:
if HF_TOKEN:
    try:
        from smolagents import InferenceClientModel, ToolCallingAgent

        reset_forest()
        model = InferenceClientModel(token=HF_TOKEN)   # дефолтная модель Inference Providers
        agent = ToolCallingAgent(
            tools=[move, gather, deposit, get_map],
            model=model,
            max_steps=12,
        )
        agent.run("Добудь одно дерево (wood) в лесу и сдай его на склад. "
                  "Действуй только инструментами; если получил ошибку — читай её code и message.")
        print()
        print("Склад после прогона:", forest.stock)
    except Exception as e:
        print("Живой прогон не прошёл -> мягкий пропуск:", repr(e))
else:
    print("Блок 3 пропущен: нет HF_TOKEN. Это ожидаемо при keyless-прогоне.")

## Блок 4 (опционально). Те же принципы против живого API

Финальный мостик к реальности — живой Cognopolis на `https://kindomklaster.com`, боевое развёртывание того же чек-листа. Совпадения с лесорубом буквальные: вместо `move(direction)` с четырьмя направлениями — `move_dir` с `enum` из восьми (добавляются диагонали); ошибки — те же обучающие пары `{code, message}` (`no_resource_here` и `inventory_full` вы уже видели); после каждого действия — кулдаун; ответ приходит тем же конвертом, только третье поле зовётся `character`, а не `state`.

Нужен токен **вашего** жителя: в игре Ратуша → вкладка «аккаунт» → «копировать»; положите его в переменную окружения `COGNOPOLIS_TOKEN`. Блок **необязательный** и делает мягкий пропуск, если сервера нет в сети или токена нет. Таймауты короткие, чтобы прогон не висел; ходим только своим персонажем.

In [ ]:
API_BASE = "https://kindomklaster.com"
LIVE_API_OK = False
try:
    import requests
    # быстрый ping карты (без auth) с коротким таймаутом
    _r = requests.get(f"{API_BASE}/map", timeout=4)
    LIVE_API_OK = _r.status_code == 200
    print("Живой API доступен:" if LIVE_API_OK else "API ответил, но не 200:",
          _r.status_code)
except Exception as e:
    print("Живой API недоступен -> Блок 4 пропущен (keyless-прогон это норма):", repr(e))

In [ ]:
import os
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # токен жителя: Ратуша -> вкладка «аккаунт» -> «копировать»
if LIVE_API_OK and TOKEN:
    try:
        import requests
        H = {"Authorization": f"Bearer {TOKEN}"}
        ch = requests.get(f"{API_BASE}/character", headers=H, timeout=6).json()
        print("Ваш персонаж на", (ch["x"], ch["y"]), "| токен задан (скрыт)")

        # направленный шаг — координат живой API не принимает, как и наш move
        direction = "east" if ch["x"] < 6 else "west"
        mv = requests.post(f"{API_BASE}/actions/move/{direction}",
                           json={"reason": "пробный шаг по живому миру"},
                           headers=H, timeout=6).json()
        print(f"move_dir({direction}) ->", mv.get("result", mv.get("error")))

        time.sleep(mv.get("cooldown", 1.0) + 0.1)
        g = requests.post(f"{API_BASE}/actions/gather",
                          json={"reason": "пробую добыть, если стою на узле"},
                          headers=H, timeout=6).json()
        # скорее всего узла на этой клетке нет — и это тоже урок: обучающая ошибка живьём
        print("gather ->", g.get("result", g.get("error")))
        print()
        print("Живой сервер отвечает тем же конвертом {result, cooldown, character} и теми же")
        print("ошибками {code, message}, что наш мок. Дизайн tools — боевой код, не учебная поделка.")
    except Exception as e:
        print("Запрос к живому API не прошёл -> мягкий пропуск:", repr(e))
elif LIVE_API_OK and not TOKEN:
    print("Блок 4 пропущен: задайте COGNOPOLIS_TOKEN (токен жителя — Ратуша -> вкладка «аккаунт»).")
else:
    print("Блок 4 пропущен: API недоступен. Это ожидаемо при keyless-прогоне без сети.")

## Задачи

Теперь почините tools руками. Все задачи работают **на моке** из Блока 1 — ключи не нужны. Каждая задача уже содержит рабочее решение-образец (ноутбук остаётся зелёным на `Run all`); ваша работа — разобрать его, изменить под себя и прогнать снова, сверяясь с критериями приёма в конце.

Подход такой: запустите готовые ячейки, убедитесь, что цифры сходятся, а затем поэкспериментируйте — поменяйте имена, тексты ошибок, значения `enum` и посмотрите, как реагируют харнесс и линтер.

### Задача 1. Переписать спрятанный god-сценарий в узкий tool

Помните тайный глагол из Блока 1? В `do_action` спрятан крафт топора: `do_action("assemble", "axe_kit")` — из 3 wood на складе. Модель об этом узнать не может в принципе: контракт `do_action` про топор молчит.

Ваш образец ниже — тот же крафт как узкий smolagents-tool `craft_axe`, по образцу `deposit`: `@tool` без аргументов, кулдаун-гвард, обучающие ошибки (тот же `not_at_storehouse` плюс новая `not_enough_wood` с точными цифрами need/have), конверт `{result, cooldown, state}`. Сравните оба на одном и том же промахе: у god-версии «не на складе» и «мало дерева» неразличимы.

Дальше — сами: добавьте по этому же образцу, например, `craft_pickaxe` из 2 stone и прогоните оба через линтер.

In [ ]:
# --- как это выглядит через god-tool (плохо) ---
reset_forest()
forest.stock = {"wood": 2}          # на складе пока только 2 wood
print("god  assemble/axe_kit ->", do_action(action="assemble", target="axe_kit"))
print("     ...invalid — а почему? не на складе? мало дерева? не тот глагол? Не узнать.")
print()


# --- образец: узкий tool по образцу deposit ---
@tool
def craft_axe() -> dict:
    """Скрафтить топор из 3 wood со склада. Работает только на клетке склада (0, 0)."""
    wait = forest.cooldown_left()
    if wait > 0:
        return {"error": {"code": "on_cooldown",
                          "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке (0, 0), а вы — на {forest.pos}. "
                                     "Дойдите до склада и повторите craft_axe."}}
    have = forest.stock.get("wood", 0)
    if have < 3:
        return {"error": {"code": "not_enough_wood",
                          "message": f"Для топора нужно 3 wood на складе, есть {have} — "
                                     "добудьте gather и сдайте deposit."}}
    forest.stock["wood"] -= 3
    forest.stock["axe"] = forest.stock.get("axe", 0) + 1
    forest.start_cooldown()
    return {"result": {"crafted": "axe", "spent": {"wood": 3}},
            "cooldown": Forest.COOLDOWN,
            "state": {"pos": list(forest.pos), "backpack": dict(forest.backpack),
                      "stock": dict(forest.stock)}}


resp = craft_axe()
print("good craft_axe (2 wood) ->", resp["error"]["code"], "—", resp["error"]["message"])
forest.stock["wood"] = 3
print("good craft_axe (3 wood) ->", craft_axe()["result"])

### Задача 2. Сделать ошибку обучающей: `unknown_recipe` с каталогом

Обобщим крафт: `craft(recipe)` с каталогом `RECIPES`. Аргумент `recipe` — не кандидат в `enum`: каталог живой, в реальной игре он растёт с каждым зданием. Это ровно случай из лекции, куда `enum` не дотягивается, — и там работает приём **«enum в форме ошибки»**: неизвестное значение получает в ответ весь список допустимых, прямо в `message`. Как `unknown option --forse; did you mean --force?` в хорошем CLI.

Ниже `craft` нарочно полуслепой: код `unknown_recipe` есть, а подсказки нет. Образец починки — `craft_v2`: сообщение перечисляет каталог с ингредиентами. Ваш ход — добавить в каталог свой рецепт и убедиться, что ошибка сама его рекламирует.

In [ ]:
RECIPES = {"axe": {"wood": 3}, "hammer": {"wood": 1, "stone": 2}}


@tool
def craft(recipe: str) -> dict:
    """Скрафтить предмет по рецепту; ингредиенты тратятся со склада.

    Args:
        recipe: имя рецепта из каталога.
    """
    # для краткости без кулдауна и проверки клетки — в бою добавьте, как в craft_axe
    need = RECIPES.get(recipe)
    if need is None:
        # ПОЛУСЛЕПАЯ ошибка: код есть, подсказки нет — чиним ниже
        return {"error": {"code": "unknown_recipe", "message": "Нет такого рецепта."}}
    have = {r: forest.stock.get(r, 0) for r in need}
    if any(have[r] < q for r, q in need.items()):
        short = ", ".join(f"{r}: нужно {q}, есть {have[r]}" for r, q in need.items())
        return {"error": {"code": "not_enough_resources",
                          "message": f"Не хватает на складе ({short}) — добудьте gather и сдайте deposit."}}
    for r, q in need.items():
        forest.stock[r] -= q
    forest.stock[recipe] = forest.stock.get(recipe, 0) + 1
    return {"result": {"crafted": recipe, "spent": dict(need)}}


def unknown_recipe_msg(recipe):
    listing = "; ".join(
        f"{name} (" + ", ".join(f"{q} {r}" for r, q in need.items()) + ")"
        for name, need in sorted(RECIPES.items()))
    return f"Неизвестный рецепт {recipe!r}. Доступны: {listing}."


@tool
def craft_v2(recipe: str) -> dict:
    """Скрафтить предмет по рецепту; неизвестный рецепт получает в ответ весь каталог.

    Args:
        recipe: имя рецепта из каталога.
    """
    need = RECIPES.get(recipe)
    if need is None:
        return {"error": {"code": "unknown_recipe",
                          "message": unknown_recipe_msg(recipe)}}   # починили!
    return craft(recipe=recipe)


reset_forest()
print("БЫЛО:  craft('sword') ->", craft(recipe="sword")["error"])
resp = craft_v2(recipe="sword")["error"]
print("СТАЛО: craft_v2('sword') ->", resp["code"])
print("       ", resp["message"])
assert "axe" in resp["message"] and "hammer" in resp["message"], \
    "каталог должен быть перечислен прямо в тексте ошибки"

### Задача 3. Идемпотентность через idempotency-key

Кулдаун — игровой приём против дубля на ретрае. В «настоящих» API ту же задачу решает `idempotency-key`: клиент шлёт уникальный ключ операции, и сервер на повтор с тем же ключом отдаёт **прежний** результат, не выполняя действие заново (так работают платёжные API).

Реализуем это поверх `gather`: повтор с тем же ключом не кладёт второе дерево в рюкзак. Заметьте разницу с кулдауном из микропроверки лекции: кулдаун отбивает ретрай **ошибкой** («подожди»), а idempotency-key молча возвращает прежний **успех** — агенту даже не нужно ничего чинить. Обёртка здесь — обычная функция; в проде ключ был бы полем `inputs` самого tool.

In [ ]:
_GATHER_SEEN = {}   # idempotency_key -> сохранённый ответ


def gather_idempotent(idempotency_key: str, resource=None) -> dict:
    """gather с idempotency-key: повтор с тем же ключом возвращает прежний ответ."""
    if idempotency_key in _GATHER_SEEN:
        cached = dict(_GATHER_SEEN[idempotency_key])   # ретрай: мир не трогаем
        cached["idempotent_replay"] = True
        return cached
    resp = gather(resource=resource)
    if "error" not in resp:
        _GATHER_SEEN[idempotency_key] = resp
    return resp


# встанем на дерево (1, 2)
reset_forest()
for d in ("east", "south", "south"):
    wait_ready()
    move(direction=d)
wait_ready()

key = "op-7f3a"   # уникальный ключ одной логической операции добычи
first = gather_idempotent(key, resource="wood")
print("первый вызов ->", first["result"], "| рюкзак:", forest.backpack)
# сеть «потеряла» ответ, фреймворк повторил с тем же ключом (кулдаун ещё идёт!):
replay = gather_idempotent(key, resource="wood")
print("повтор       ->", replay["result"], "| replay:", replay.get("idempotent_replay"),
      "| рюкзак:", forest.backpack)
assert forest.backpack == {"wood": 1}, "повтор с тем же ключом не должен добывать второе дерево"
print("OK: дубля нет — и вместо ошибки on_cooldown ретрай получил прежний успех.")

### Задача 4. Догнать «худую» спеку до 8/8

Соберём всё вместе. Ниже два настоящих smolagents-tool с одним именем `chop_tree`: версия «до» проседает почти по всем буквам (худой description, свободная строка вместо `enum`, `{"error": "fail"}`, мутация без кулдауна, безлимитный список событий в ответе), версия «после» — починена: описание с «когда звать», `enum`-ожидание, а логику, ошибки, конверт и кулдаун она честно переиспользует, делегируя проверенному `gather`.

Прогоните линтер, зафиксируйте «до/после» и поэкспериментируйте: верните какую-нибудь правку назад — например, сотрите `enum` из `inputs` — и посмотрите, какая буква упадёт.

In [ ]:
class ChopTreeV1(Tool):
    name = "chop_tree"
    description = "Срубить дерево."                       # худая заглушка -> B мимо
    inputs = {
        "tree": {"type": "string",
                 "description": "какое дерево рубить"},   # закрытый по смыслу список без enum -> C мимо
    }
    output_type = "object"

    def forward(self, tree: str) -> dict:
        if forest.nodes.get(forest.pos) != "wood":
            return {"error": "fail"}                      # немая ошибка -> E мимо
        forest.backpack["wood"] = forest.backpack.get("wood", 0) + 1   # без кулдауна -> F мимо
        return {"ok": True, "log": ["swing", "swing", "crack", "..."]}  # проза/безлимит -> D, H мимо


class ChopTreeV2(Tool):
    name = "chop_tree"
    description = (
        "Срубить одно дерево на клетке под ногами и положить wood в рюкзак. "
        "Зовите, стоя на клетке с деревом (найдите её get_map и дойдите move); "
        "побочный эффект — кулдаун."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood"],
            "nullable": True,
            "description": "Ожидаемый ресурс; у топора он один — wood.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        # делегируем проверенному gather: его ошибки, конверт и кулдаун — наши
        return gather(resource=resource or "wood")


meta_chop_before = {
    "closed_set_args": ["tree"],
    "returns_envelope": False, "next_step_fields": [],
    "error_has_code": False, "error_has_message": False,
    "is_mutation": True, "idempotency": None,
    "returns_list": True, "has_limit": False,
}
meta_chop_after = {
    "closed_set_args": ["resource"],
    "returns_envelope": True, "next_step_fields": ["cooldown", "state"],
    "error_has_code": True, "error_has_message": True,
    "is_mutation": True, "idempotency": "cooldown",
    "returns_list": False, "has_limit": False,
}

print("=== chop_tree ДО ===")
rb = report("chop_tree (before)", ChopTreeV1(), meta_chop_before)
print()
print("=== chop_tree ПОСЛЕ ===")
ra = report("chop_tree (after)", ChopTreeV2(), meta_chop_after)
print()
TARGET = 8
print(f"score: {rb['score']} -> {ra['score']} (цель {TARGET}/8)")
assert ra["score"] >= TARGET, "после правок tool должен набрать целевой score 8/8"
print("Цель достигнута: tool прошёл весь чек-лист A->H.")

## Что дальше

Вы прошли путь от «тупящего» агента до починенного интерфейса: расщепили god-tool на инструменты по одной цели, поставили `enum`-рельсы в схему, написали обучающие ошибки, завернули ответы в конверт, сделали мутации идемпотентными и прогнали линтер по A→H. Контракт при этом ни разу не зависел от движка: те же четыре объекта читал и скриптовый харнесс, и `ToolCallingAgent`, и (в Блоке 4) их двойники в живой игре.

Следующий шаг — увидеть, что контракт переживает и смену движка целиком: Модуль 11.7 «Tools в движках» (на сайте курса) — тот же лесоруб на голом Anthropic API, JSON Schema руками, `tool_choice` и параллельные вызовы. Дальше Модуль 13.6 «Skills» — лесоруб выучит процедуру «supply run» поверх тех же `move` / `gather` / `deposit`, и Модуль 14.5 «MCP» — как один контракт отдать по стандартному протоколу любому агенту.

**Критерии приёма (проверяете сами):**

- ноутбук прогнан целиком (`Run all`) keyless без ошибок;
- спрятанный god-сценарий переписан в узкий tool по одной цели с обучающими ошибками (Задача 1);
- `unknown_recipe` перечисляет каталог рецептов прямо в тексте ошибки (Задача 2);
- мутация идемпотентна — повтор с тем же ключом не дублирует эффект (Задача 3);
- линтер по A→H показывает рост score «до/после» и достигает 8/8 (Задача 4).

Если так — домашка сдана, преподаватель не нужен.